In [497]:
import pandas as pd

In [498]:
rental_school_hospital = pd.read_csv("../data/curated/rental_data_distance_sf.csv")
rental_crime = pd.read_csv("../data/curated/rental_data_crime_join.csv")
postcode_2011 = pd.read_csv("../data/curated/postcode_info_2011.csv")
postcode_2016 = pd.read_csv("../data/curated/postcode_info_2011.csv")
postcode_2021 = pd.read_csv("../data/curated/postcode_info_2011.csv")


In [499]:
# lowercase column names
postcode_2011.columns = postcode_2011.columns.str.lower()
postcode_2016.columns = postcode_2016.columns.str.lower()
postcode_2021.columns = postcode_2021.columns.str.lower()
rental_crime.columns = rental_crime.columns.str.lower()
rental_school_hospital.columns = rental_school_hospital.columns.str.lower()

In [500]:
# drop duplicate properties
rental_school_hospital = rental_school_hospital.drop_duplicates(
    subset=["name", "cost_text", "coordinates", "property type", "beds", "baths", "parking", 
            ])

rental_crime = rental_crime.drop_duplicates(
    subset=["name", "cost_text", "coordinates", "property type", "beds", "baths", "parking"])

In [501]:
# lowercase column names
postcode_2011.columns = postcode_2011.columns.str.lower()
postcode_2016.columns = postcode_2016.columns.str.lower()
postcode_2021.columns = postcode_2021.columns.str.lower()


# add year in column name for postcode info dataframes
postcode_2011.columns = [col + " 2011" for col in postcode_2011.columns]
postcode_2016.columns = [col + " 2016" for col in postcode_2016.columns]
postcode_2021.columns = [col + " 2021" for col in postcode_2021.columns]

In [503]:
# impute missing values as 0
rental_crime["offence count 2023"] = rental_crime["offence count 2023"].fillna(0)
rental_crime["offence count 2022"] = rental_crime["offence count 2022"].fillna(0)
rental_crime["offence count 2021"] = rental_crime["offence count 2021"].fillna(0)

# Merge rental_data_school_hopsital with rental_crime

In [505]:
print(len(rental_school_hospital))
rental_merged = pd.merge(rental_school_hospital, rental_crime, on=["name", "property type", "coordinates", "weekly rent amount",
                                                                   "beds", "baths", "parking", "title"], how="inner", 
                         suffixes=('_left', '_right'))

# drop columns that are duplicates
for col_name in rental_merged.columns:
    col_name_list = col_name.split("_")
    if col_name_list[-1] == "right":
        rental_merged = rental_merged.drop(col_name, axis=1)
    # rename column name to original
    if col_name_list[-1] == "left":
        rental_merged.rename(columns={col_name: "_".join(col_name_list[0:-1])}, inplace=True)
        
print(len(rental_merged))
rental_merged.columns

11335
11341


Index(['name', 'cost_text', 'coordinates', 'property type', 'beds', 'baths',
       'parking', 'title', 'description', 'property features', 'nbn',
       'weekly rent amount', 'postcode', 'geometry', 'sa2_code21',
       'sa2_name21', 'closest primary school', 'distance primary school',
       'closest secondary school', 'distance secondary school',
       'closest university campus', 'distance university campus',
       'offence count 2023', 'offence count 2022', 'offence count 2021'],
      dtype='object')

# Merge different years' postcode data together 

In [506]:
# merge 2011 and 2016
postcode_merged = pd.merge(postcode_2011, postcode_2016, left_on = "postcode 2011", right_on = "postcode 2016",
                           how="inner")
postcode_merged = postcode_merged.drop("postcode 2011", axis=1)

postcode_merged

,estimated resident population (no.) 2011,employed (no.) 2011,unemployed (no.) 2011,in the labour force (no.) 2011,total persons aged 15 years and over (no.) 2011,land area (ha) 2011,total families (no.) 2011,total dwellings (no.) 2011,median equivalised total household income (weekly) ($) 2011,median weekly household rental payment ($) 2011,...,employed (no.) 2016,unemployed (no.) 2016,in the labour force (no.) 2016,total persons aged 15 years and over (no.) 2016,land area (ha) 2016,total families (no.) 2016,total dwellings (no.) 2016,median equivalised total household income (weekly) ($) 2016,median weekly household rental payment ($) 2016,average family size (no. of persons) 2016
0,0.0,55833.0,4270.0,60108.0,97390.0,0.0,23110.0,0.0,1017.090909,402.090909,...,55833.0,4270.0,60108.0,97390.0,0.0,23110.0,0.0,1017.090909,402.090909,2.481818
1,0.0,3122.0,109.0,3234.0,4630.0,0.0,1087.0,0.0,1504.000000,420.000000,...,3122.0,109.0,3234.0,4630.0,0.0,1087.0,0.0,1504.000000,420.000000,2.300000
2,0.0,7840.0,706.0,8558.0,13921.0,0.0,3015.0,0.0,1014.000000,395.000000,...,7840.0,706.0,8558.0,13921.0,0.0,3015.0,0.0,1014.000000,395.000000,2.500000
3,0.0,38469.0,2536.0,41011.0,63808.0,0.0,15049.0,0.0,1120.000000,417.625000,...,38469.0,2536.0,41011.0,63808.0,0.0,15049.0,0.0,1120.000000,417.625000,2.387500
4,0.0,12197.0,696.0,12899.0,19496.0,0.0,4936.0,0.0,1257.000000,483.500000,...,12197.0,696.0,12899.0,19496.0,0.0,4936.0,0.0,1257.000000,483.500000,2.350000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,0.0,4020.0,172.0,4192.0,6995.0,0.0,2434.0,0.0,630.000000,195.000000,...,4020.0,172.0,4192.0,6995.0,0.0,2434.0,0.0,630.000000,195.000000,3.000000
423,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,2.800000
424,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,2.800000
425,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,2.800000


In [507]:
# merge 2021 data with rest of postcode data
postcode_merged = pd.merge(postcode_merged, postcode_2021, left_on = "postcode 2016", right_on = "postcode 2021",
                           how="inner")
postcode_merged = postcode_merged.drop("postcode 2016", axis=1)
postcode_merged

,estimated resident population (no.) 2011,employed (no.) 2011,unemployed (no.) 2011,in the labour force (no.) 2011,total persons aged 15 years and over (no.) 2011,land area (ha) 2011,total families (no.) 2011,total dwellings (no.) 2011,median equivalised total household income (weekly) ($) 2011,median weekly household rental payment ($) 2011,...,employed (no.) 2021,unemployed (no.) 2021,in the labour force (no.) 2021,total persons aged 15 years and over (no.) 2021,land area (ha) 2021,total families (no.) 2021,total dwellings (no.) 2021,median equivalised total household income (weekly) ($) 2021,median weekly household rental payment ($) 2021,average family size (no. of persons) 2021
0,0.0,55833.0,4270.0,60108.0,97390.0,0.0,23110.0,0.0,1017.090909,402.090909,...,55833.0,4270.0,60108.0,97390.0,0.0,23110.0,0.0,1017.090909,402.090909,2.481818
1,0.0,3122.0,109.0,3234.0,4630.0,0.0,1087.0,0.0,1504.000000,420.000000,...,3122.0,109.0,3234.0,4630.0,0.0,1087.0,0.0,1504.000000,420.000000,2.300000
2,0.0,7840.0,706.0,8558.0,13921.0,0.0,3015.0,0.0,1014.000000,395.000000,...,7840.0,706.0,8558.0,13921.0,0.0,3015.0,0.0,1014.000000,395.000000,2.500000
3,0.0,38469.0,2536.0,41011.0,63808.0,0.0,15049.0,0.0,1120.000000,417.625000,...,38469.0,2536.0,41011.0,63808.0,0.0,15049.0,0.0,1120.000000,417.625000,2.387500
4,0.0,12197.0,696.0,12899.0,19496.0,0.0,4936.0,0.0,1257.000000,483.500000,...,12197.0,696.0,12899.0,19496.0,0.0,4936.0,0.0,1257.000000,483.500000,2.350000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,0.0,4020.0,172.0,4192.0,6995.0,0.0,2434.0,0.0,630.000000,195.000000,...,4020.0,172.0,4192.0,6995.0,0.0,2434.0,0.0,630.000000,195.000000,3.000000
423,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,2.800000
424,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,2.800000
425,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,2.800000


# Merge postcode data with rental data

In [508]:
print(len(rental_merged))
rental_merged = pd.merge(rental_merged, postcode_merged, left_on = "postcode", right_on = "postcode 2021", 
                              how="left")
rental_merged = rental_merged.drop("postcode 2021", axis=1)
print(len(rental_merged))
rental_merged

11341
11341


,name,cost_text,coordinates,property type,beds,baths,parking,title,description,property features,...,employed (no.) 2021,unemployed (no.) 2021,in the labour force (no.) 2021,total persons aged 15 years and over (no.) 2021,land area (ha) 2021,total families (no.) 2021,total dwellings (no.) 2021,median equivalised total household income (weekly) ($) 2021,median weekly household rental payment ($) 2021,average family size (no. of persons) 2021
0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...","['Furnished*', 'Broadband internet access*', '...",...,20267.0,934.0,21200.0,33991.0,0.0,10901.0,0.0,1089.0,367.00,3.133333
1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,"['Gas*', 'Internal Laundry*', 'Secure Parking*...",...,25316.0,1259.0,26567.0,43134.0,0.0,14051.0,0.0,729.5,269.25,2.925000
2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,[],...,3626.0,200.0,3823.0,7615.0,0.0,2547.0,0.0,628.0,271.00,2.700000
3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,"['Built in wardrobes', 'Internal Laundry', 'Ga...",...,25316.0,1259.0,26567.0,43134.0,0.0,14051.0,0.0,729.5,269.25,2.925000
4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,"['Ensuite', 'Furnished', 'Built in wardrobes*'...",...,25316.0,1259.0,26567.0,43134.0,0.0,14051.0,0.0,729.5,269.25,2.925000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11336,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...","['Study', 'Swimming Pool', 'Ducted Cooling', '...",...,17761.0,743.0,18496.0,30688.0,0.0,10137.0,0.0,1123.5,402.00,3.050000
11337,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,"['Ensuite', 'Deck', 'Dishwasher', 'Ducted Heat...",...,2174.0,181.0,2356.0,5612.0,0.0,1855.0,0.0,543.0,240.00,2.600000
11338,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",[],...,12197.0,696.0,12899.0,19496.0,0.0,4936.0,0.0,1257.0,483.50,2.350000
11339,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,"['Air conditioning*', 'Built in wardrobes*', '...",...,16540.0,845.0,17386.0,29273.0,0.0,9566.0,0.0,987.0,397.50,3.150000


In [509]:
rental_merged.isna().any()

name                                                           False
cost_text                                                      False
coordinates                                                    False
property type                                                  False
beds                                                           False
baths                                                          False
parking                                                        False
title                                                          False
description                                                    False
property features                                              False
nbn                                                            False
weekly rent amount                                             False
postcode                                                       False
geometry                                                       False
sa2_code21                        

# Merge town data with rental data

In [510]:
town_data = pd.read_csv("../data/curated/rental_data_town_join.csv")
# lowercase column names
town_data.columns = town_data.columns.str.lower()
print(town_data.columns)
# drop duplicate rows
print(len(town_data))
town_data = town_data.drop_duplicates(
    subset=["name", "cost_text", "coordinates", "property type", "beds", "baths", "parking"])
print(len(town_data))
town_data

Index(['unnamed: 0', 'name', 'cost_text', 'coordinates', 'property type',
       'beds', 'baths', 'parking', 'title', 'description', 'property features',
       'nbn', 'weekly rent amount', 'postcode', 'geometry', 'sa2_code21',
       'sa2_name21', 'total_supermarkets', 'major_supermarkets', 'city_centre',
       'closest_city_centre', 'coastal', 'closest_cc_dist', 'dist_to_melb'],
      dtype='object')
11409
11335


,unnamed: 0,name,cost_text,coordinates,property type,beds,baths,parking,title,description,...,geometry,sa2_code21,sa2_name21,total_supermarkets,major_supermarkets,city_centre,closest_city_centre,coastal,closest_cc_dist,dist_to_melb
0,0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...",...,POINT (145.1114189 -37.8353315),207031167.0,Surrey Hills (East) - Mont Albert,2.0,0.0,False,Melbourne,False,11.566742,12.618789
1,1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,...,POINT (144.3060777 -38.1962932),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
2,2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,...,POINT (145.0045603 -38.3138012),214021377.0,Dromana,1.0,0.0,False,Cranbourne,False,37.174742,58.248165
3,3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,...,POINT (144.3154857 -38.19508),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
4,4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,...,POINT (144.3107906 -38.1845196),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11404,11404,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...",...,POINT (144.9997396 -37.9044002),208011169.0,Brighton (Vic.),8.0,0.0,False,Melbourne,False,7.399952,10.876836
11405,11405,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,...,POINT (144.7147899 -38.1439923),203031051.0,Portarlington,2.0,0.0,False,Geelong,True,27.298679,44.708244
11406,11406,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",...,POINT (144.9646982 -37.8213342),206041508.0,Southbank (West) - South Wharf,1.0,0.0,False,Melbourne,False,1.775904,1.827457
11407,11407,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,...,POINT (145.0775507 -37.8117577),207011147.0,Balwyn,4.0,0.0,False,Melbourne,False,9.486256,10.159737


In [511]:
rental_merged

,name,cost_text,coordinates,property type,beds,baths,parking,title,description,property features,...,employed (no.) 2021,unemployed (no.) 2021,in the labour force (no.) 2021,total persons aged 15 years and over (no.) 2021,land area (ha) 2021,total families (no.) 2021,total dwellings (no.) 2021,median equivalised total household income (weekly) ($) 2021,median weekly household rental payment ($) 2021,average family size (no. of persons) 2021
0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...","['Furnished*', 'Broadband internet access*', '...",...,20267.0,934.0,21200.0,33991.0,0.0,10901.0,0.0,1089.0,367.00,3.133333
1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,"['Gas*', 'Internal Laundry*', 'Secure Parking*...",...,25316.0,1259.0,26567.0,43134.0,0.0,14051.0,0.0,729.5,269.25,2.925000
2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,[],...,3626.0,200.0,3823.0,7615.0,0.0,2547.0,0.0,628.0,271.00,2.700000
3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,"['Built in wardrobes', 'Internal Laundry', 'Ga...",...,25316.0,1259.0,26567.0,43134.0,0.0,14051.0,0.0,729.5,269.25,2.925000
4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,"['Ensuite', 'Furnished', 'Built in wardrobes*'...",...,25316.0,1259.0,26567.0,43134.0,0.0,14051.0,0.0,729.5,269.25,2.925000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11336,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...","['Study', 'Swimming Pool', 'Ducted Cooling', '...",...,17761.0,743.0,18496.0,30688.0,0.0,10137.0,0.0,1123.5,402.00,3.050000
11337,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,"['Ensuite', 'Deck', 'Dishwasher', 'Ducted Heat...",...,2174.0,181.0,2356.0,5612.0,0.0,1855.0,0.0,543.0,240.00,2.600000
11338,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",[],...,12197.0,696.0,12899.0,19496.0,0.0,4936.0,0.0,1257.0,483.50,2.350000
11339,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,"['Air conditioning*', 'Built in wardrobes*', '...",...,16540.0,845.0,17386.0,29273.0,0.0,9566.0,0.0,987.0,397.50,3.150000


In [512]:
rental_merged = pd.merge(rental_merged, town_data, on=["name", "property type", "coordinates", "weekly rent amount",
                                                                   "beds", "baths", "parking", "title"], how="inner", 
                         suffixes=('_left', '_right'))

# drop columns that are duplicates
for col_name in rental_merged.columns:
    col_name_list = col_name.split("_")
    if col_name_list[-1] == "right":
        rental_merged = rental_merged.drop(col_name, axis=1)
    # rename column name to original
    if col_name_list[-1] == "left":
        rental_merged.rename(columns={col_name: "_".join(col_name_list[0:-1])}, inplace=True)

In [513]:
rental_merged = rental_merged.drop(["title", "description", "cost_text"], axis=1)
print(list(rental_merged.columns))

['name', 'coordinates', 'property type', 'beds', 'baths', 'parking', 'property features', 'nbn', 'weekly rent amount', 'postcode', 'geometry', 'sa2_code21', 'sa2_name21', 'closest primary school', 'distance primary school', 'closest secondary school', 'distance secondary school', 'closest university campus', 'distance university campus', 'offence count 2023', 'offence count 2022', 'offence count 2021', 'estimated resident population (no.) 2011', 'employed (no.) 2011', 'unemployed (no.) 2011', 'in the labour force (no.) 2011', 'total persons aged 15 years and over (no.) 2011', 'land area (ha) 2011', 'total families (no.) 2011', 'total dwellings (no.) 2011', 'median equivalised total household income (weekly) ($) 2011', 'median weekly household rental payment ($) 2011', 'average family size (no. of persons) 2011', 'estimated resident population (no.) 2016', 'employed (no.) 2016', 'unemployed (no.) 2016', 'in the labour force (no.) 2016', 'total persons aged 15 years and over (no.) 2016',

# Merge rental data with supermarket data

In [514]:
supermarket_data = pd.read_csv("../data/curated/supermarket_data.csv")
supermarket_data.columns = supermarket_data.columns.str.lower()
# convert postcode to float
supermarket_data["postcode"] = pd.to_numeric(supermarket_data["postcode"], errors="coerce")
print(len(supermarket_data))
# drop non-numeric postcodes
print(len(supermarket_data[supermarket_data["postcode"].isna()]))
supermarket_data = supermarket_data.dropna(subset = ["postcode"])
print(len(supermarket_data))

344
0
344


In [515]:
print(len(rental_merged))

11353


In [516]:
supermarket_data[supermarket_data["postcode"] == 3216]

,postcode,total_supermarkets,major_supermarkets
159,3216,14,0


In [517]:
supermarket_data["postcode"].value_counts()

postcode
3000    1
3523    1
3556    1
3555    1
3551    1
       ..
3155    1
3153    1
3152    1
3151    1
3995    1
Name: count, Length: 344, dtype: int64

In [518]:
# merge rows with same postcode but different town names together
supermarket_data = supermarket_data.groupby("postcode").sum().reset_index()
supermarket_data.value_counts()

postcode  total_supermarkets  major_supermarkets
3000      11                  0                     1
3523      2                   0                     1
3556      1                   0                     1
3555      6                   0                     1
3551      1                   0                     1
                                                   ..
3155      6                   0                     1
3153      5                   0                     1
3152      4                   0                     1
3151      3                   0                     1
3995      3                   0                     1
Name: count, Length: 344, dtype: int64

In [519]:
supermarket_data[supermarket_data["postcode"] == 3216]

,postcode,total_supermarkets,major_supermarkets
159,3216,14,0


In [520]:
# merge rental data and supermarket data
rental_merged = pd.merge(rental_merged, supermarket_data, how = "left", on="postcode")


In [521]:
rental_merged

,name,coordinates,property type,beds,baths,parking,property features,nbn,weekly rent amount,postcode,...,unnamed: 0,total_supermarkets_x,major_supermarkets_x,city_centre,closest_city_centre,coastal,closest_cc_dist,dist_to_melb,total_supermarkets_y,major_supermarkets_y
0,Rooms/227 Elgar Road Surrey Hills VIC 3127,"[-37.8353315, 145.1114189]",House,1,2,6,"['Furnished*', 'Broadband internet access*', '...",No NBN,120,3127,...,0,2.0,0.0,False,Melbourne,False,11.566742,12.618789,2.0,0.0
1,30 Daly Boulevard Highton VIC 3216,"[-38.1962932, 144.3060777]",House,9,2,2,"['Gas*', 'Internal Laundry*', 'Secure Parking*...",nbn® Fibre to the Node (FTTN) is available in ...,130,3216,...,1,14.0,0.0,False,Geelong,False,7.459818,71.172437,14.0,0.0
2,Berth 5274 Martha Cove Waterway Safety Beach V...,"[-38.3138012, 145.0045603]",House,1,1,0,[],No NBN,140,3936,...,2,1.0,0.0,False,Cranbourne,False,37.174742,58.248165,1.0,0.0
3,18 Cedarville Close Highton VIC 3216,"[-38.19508, 144.3154857]",House,1,2,6,"['Built in wardrobes', 'Internal Laundry', 'Ga...",nbn® Fibre to the Premises (FTTP) is available...,140,3216,...,3,14.0,0.0,False,Geelong,False,7.459818,71.172437,14.0,0.0
4,123 Thornhill Road Highton VIC 3216,"[-38.1845196, 144.3107906]",House,1,4,3,"['Ensuite', 'Furnished', 'Built in wardrobes*'...",nbn® Fibre to the Node (FTTN) is available in ...,150,3216,...,4,14.0,0.0,False,Geelong,False,7.459818,71.172437,14.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11348,Enquire For Address Brighton VIC 3186,"[-37.9044002, 144.9997396]",House,4,4,2,"['Study', 'Swimming Pool', 'Ducted Cooling', '...",No NBN,4400,3186,...,11404,8.0,0.0,False,Melbourne,False,7.399952,10.876836,8.0,0.0
11349,335 The Esplanade Indented Head VIC 3223,"[-38.1439923, 144.7147899]",House,4,3,0,"['Ensuite', 'Deck', 'Dishwasher', 'Ducted Heat...",No NBN,4700,3223,...,11405,2.0,0.0,False,Geelong,True,27.298679,44.708244,2.0,0.0
11350,8701/7 Riverside Quay Southbank VIC 3006,"[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,[],nbn® Fibre to the Building (FTTB) is available...,5000,3006,...,11406,1.0,0.0,False,Melbourne,False,1.775904,1.827457,1.0,0.0
11351,1 Weston Street Balwyn VIC 3103,"[-37.8117577, 145.0775507]",House,6,7,8,"['Air conditioning*', 'Built in wardrobes*', '...",nbn® Hybrid Fibre Coaxial (HFC) is available i...,9950,3103,...,11407,4.0,0.0,False,Melbourne,False,9.486256,10.159737,4.0,0.0


In [522]:
rental_merged.isna().any()

name                    False
coordinates             False
property type           False
beds                    False
baths                   False
                        ...  
coastal                  True
closest_cc_dist          True
dist_to_melb             True
total_supermarkets_y     True
major_supermarkets_y     True
Length: 65, dtype: bool

In [523]:
print(len(rental_merged))
# drop rows with null value for supermarkets after merge
rental_merged = rental_merged.dropna(subset=["total"])
print(len(rental_merged))

11353


KeyError: ['total']

In [ ]:
rental_merged.isna().any()

name                                                           False
coordinates                                                    False
property type                                                  False
beds                                                           False
baths                                                          False
parking                                                        False
property features                                              False
nbn                                                            False
weekly rent amount                                             False
postcode                                                       False
geometry                                                       False
sa2_code21                                                      True
sa2_name21                                                      True
closest primary school                                         False
distance primary school           

In [ ]:
rental_merged.to_csv("../data/curated/merged_data.csv", index=False)

In [ ]:
town_data = pd.read_csv("../data/curated/rental_data_town_join.csv")
town_data

,Unnamed: 0,name,cost_text,coordinates,property type,beds,baths,parking,title,description,...,geometry,SA2_CODE21,SA2_NAME21,Total_Supermarkets,Major_Supermarkets,city_centre,closest_city_centre,coastal,closest_cc_dist,dist_to_melb
0,0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...",...,POINT (145.1114189 -37.8353315),207031167.0,Surrey Hills (East) - Mont Albert,2.0,0.0,False,Melbourne,False,11.566742,12.618789
1,1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,...,POINT (144.3060777 -38.1962932),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
2,2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,...,POINT (145.0045603 -38.3138012),214021377.0,Dromana,1.0,0.0,False,Cranbourne,False,37.174742,58.248165
3,3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,...,POINT (144.3154857 -38.19508),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
4,4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,...,POINT (144.3107906 -38.1845196),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11404,11404,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...",...,POINT (144.9997396 -37.9044002),208011169.0,Brighton (Vic.),8.0,0.0,False,Melbourne,False,7.399952,10.876836
11405,11405,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,...,POINT (144.7147899 -38.1439923),203031051.0,Portarlington,2.0,0.0,False,Geelong,True,27.298679,44.708244
11406,11406,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",...,POINT (144.9646982 -37.8213342),206041508.0,Southbank (West) - South Wharf,1.0,0.0,False,Melbourne,False,1.775904,1.827457
11407,11407,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,...,POINT (145.0775507 -37.8117577),207011147.0,Balwyn,4.0,0.0,False,Melbourne,False,9.486256,10.159737
